# 期末專題：為 Netflix 綠燈委員會駭客松打造特徵工程

本筆記本是 Canvas 上期末專題說明（*The Netflix Writers Dataset Pipeline*）的配套教材。該模組
說明的是 `netflix.titles.composite.csv`
**如何**被建立出來，而本筆記本要談的是它建立完成**之後**該做什麼：把
原始欄位轉換成模型——或編劇——真正能派上用場的預測因子。

下面的每個範例都是一個小型、通用的函式。它們都盡量不與某個特定欄位名稱綁死，
因此你可以在整個專題中重複使用同樣的模式。每個函式之後都會附上簡短說明，
解釋當你想預測一部新電視劇概念是否具有爆紅潛力時，*為什麼*這個因子很重要。

這份綜合資料集（composite dataset）中，每一列對應一部成功比對到的 Netflix 片名，
包含以下欄位：

| 欄位 | 意義 |
|---|---|
| `netflix_title` | Netflix Top 10 資料中出現的片名 |
| `netflix_viewing_hours` | 該片名在 Top 10 榜單期間的總觀看時數 |
| `netflix_weeks` | 該片名停留在 Top 10 榜單上的週數 |
| `netflix_year_hint` | 該片名首次出現在榜單上的年份 |
| `tmdb_title`、`tmdb_popularity`、`tmdb_vote_average`、`tmdb_vote_count` | 比對到的 TMDB 中繼資料 |
| `imdb_title`、`imdb_averageRating`、`imdb_numVotes` | 比對到的 IMDb 中繼資料 |

某一列可能會缺少 `tmdb_*` 或 `imdb_*` 的值——因為模組 2 中的模糊比對
（fuzzy-matching）流程並非對每個片名都能找到有把握的比對結果，所以
實際涵蓋率永遠不會是 100%。


## 1. 載入綜合資料集

在你自己的環境中，只需要這樣做：

```python
df = pd.read_csv("netflix/data/netflix.titles.composite.csv")
```

要產生真正的檔案，需要跑完整套 `make run` 流程（Kaggle + IMDb
下載），這需要 5 到 15 分鐘，還需要這個沙盒環境沒有的憑證。因此下面的每個
範例都是針對一份小型的**合成**（synthetic）替代資料執行，這份資料具有相同欄位，
也具有同樣的怪異之處——缺漏的比對結果、偏斜的投票數、
只靠少數幾票撐起的評分——所以程式碼是真正被實際跑過的，而不只是紙上談兵。
換上你自己的真實檔案後，其餘一切都能照常運作。


In [32]:
import numpy as np
import pandas as pd

################################################################
# rng = np.random.default_rng(7)
# n = 300

# titles = [f"Show {i:03d}" for i in range(n)]
# netflix_weeks = rng.integers(1, 14, size=n)
# netflix_viewing_hours = (netflix_weeks * rng.uniform(2_000_000, 9_000_000, size=n)).round(0)
# netflix_year_hint = rng.integers(2021, 2026, size=n)

# # 並非每個片名都能得到有把握的 TMDB / IMDb 比對結果——這裡模擬這種情況。
# has_tmdb = rng.random(n) > 0.15
# has_imdb = rng.random(n) > 0.10

# df = pd.DataFrame({
#     "netflix_title": titles,
#     "netflix_viewing_hours": netflix_viewing_hours,
#     "netflix_weeks": netflix_weeks,
#     "netflix_year_hint": netflix_year_hint,
#     "tmdb_title": np.where(has_tmdb, titles, None),
#     "tmdb_popularity": np.where(has_tmdb, rng.uniform(5, 400, size=n).round(1), np.nan),
#     "tmdb_vote_average": np.where(has_tmdb, rng.uniform(4.5, 9.0, size=n).round(1), np.nan),
#     "tmdb_vote_count": np.where(has_tmdb, rng.integers(20, 12_000, size=n), np.nan),
#     "imdb_title": np.where(has_imdb, titles, None),
#     "imdb_averageRating": np.where(has_imdb, rng.uniform(4.0, 9.3, size=n).round(1), np.nan),
#     "imdb_numVotes": np.where(has_imdb, rng.integers(15, 300_000, size=n), np.nan),
# })
################################################################

df = pd.read_csv("../netflix/data/netflix.titles.composite.csv")

df.shape, df.head(3)

((691, 13),
         key  netflix_viewing_hours  netflix_weeks  netflix_year_hint  \
 0        10               26580000              2               2021   
 1  10_fr_mi               13590000              2               2021   
 2     10_lo               12600000              1               2021   
 
                netflix_title        netflix_clean_title  \
 0                    The 100                    the 100   
 1  1000 Miles from Christmas  1000 miles from christmas   
 2                   Love 101                   love 101   
 
                tmdb_title  tmdb_popularity  tmdb_vote_average  \
 0                 The 100          127.224              7.916   
 1  100 Miles From Nowhere            1.400              3.600   
 2                Love 101           14.518              7.974   
 
    tmdb_vote_count                 imdb_title  imdb_averageRating  \
 0           7666.0                   The 100!                 NaN   
 1              5.0  1000 Miles from Christmas

## 2. 在動手做特徵工程之前，先檢查比對涵蓋率

每一個依賴 `tmdb_*` 或 `imdb_*` 欄位的下游特徵，都會繼承模糊比對步驟
所達成的涵蓋率。務必*先*檢查這一點——如果只有 40% 的片名成功比對到
IMDb，那麼任何只根據 IMDb 評分建立的特徵，就只能描述 40% 的訓練資料，
而用這種資料訓練出的模型，在剩下的資料上會悄悄地表現變差。

這一點也和駭客松直接相關：模組 2 的第 9 節揭露了 `fetch_imdb.py` 中的一個
實際錯誤——`title_key` 表達式裡 `REGEXP_REPLACE` 在 `LOWER` 之前執行，
悄悄地去掉了大寫字母，降低了比對成功率（`"Stranger Things"` 變成
`"tranger hings"`）。修正這個錯誤會直接提升下方儲存格中的數字——
每次調整過流程之後，都值得檢查一下。


In [2]:
def match_coverage(frame: pd.DataFrame, source_columns: dict[str, str]) -> pd.DataFrame:
    """回報每個來源有多少比例的列成功比對到非空值。

    Args:
        frame: 綜合資料集。
        source_columns: 把「人類可讀的來源名稱」對應到該來源中
            用來檢查是否為空值的一個欄位，例如 {"tmdb": "tmdb_title"}。

    Returns:
        一個小型 DataFrame，每個來源一列：比對成功數、總數，
        以及涵蓋率百分比。
    """
    rows = []
    total = len(frame)
    for source_name, column in source_columns.items():
        matched = frame[column].notna().sum()
        rows.append({"source": source_name, "matched": matched, "total": total,
                      "coverage_pct": round(100 * matched / total, 1)})
    return pd.DataFrame(rows)


match_coverage(df, {"tmdb": "tmdb_title", "imdb": "imdb_title"})

,source,matched,total,coverage_pct
0,tmdb,243,300,81.0
1,imdb,266,300,88.7


## 3. 定義預測目標：什麼樣的表現才算「爆紅」？

在設計預測因子之前，得先決定它們究竟要預測什麼。一個簡單、通用的做法是：
把落在某個表現欄位——這裡是 `netflix_viewing_hours`——最高分位數（quantile）
的片名，判定為爆紅（hit）。這樣就能得到一個用於分類的二元標籤，
同時保留原始欄位，供你日後想改做迴歸（預測數值而非類別）時使用。


In [3]:
def label_hits(frame: pd.DataFrame, column: str, quantile: float = 0.75) -> pd.DataFrame:
    """新增一個 `is_hit` 欄位：若 `column` 達到或超過給定的分位數則為 True。

    刻意寫成通用形式——今天可以用在觀看時數，日後換成任何其他
    數值型的表現欄位也一樣能用。
    """
    frame = frame.copy()
    threshold = frame[column].quantile(quantile)
    frame["is_hit"] = frame[column] >= threshold
    return frame


df = label_hits(df, "netflix_viewing_hours", quantile=0.75)
df["is_hit"].value_counts()

is_hit
False    225
True      75
Name: count, dtype: int64

## 4. 追劇速度（binge velocity）：榜單期間每週觀看時數

兩部劇可能累積出同樣的總觀看時數，途徑卻大不相同：一部在兩週內被觀眾
一口氣看完，另一部則溫吞地拖了十週。把總時數除以上榜週數，就能把這種差異
呈現成單一數字——這是口碑擴散速度快慢的粗略代理指標。對編劇而言，
這暗示著節奏感與「一口氣追完」的特質（短篇章節構、強力的集末鉤子）
可能和整體品質同樣重要。


In [4]:
def binge_velocity(frame: pd.DataFrame,
                    hours_col: str = "netflix_viewing_hours",
                    weeks_col: str = "netflix_weeks") -> pd.Series:
    """在 Top 10 榜單期間，平均每週的觀看時數。

    對 weeks_col == 0 的片名做除以零的防護。
    """
    weeks = frame[weeks_col].replace(0, np.nan)
    return (frame[hours_col] / weeks).rename("binge_velocity")


df["binge_velocity"] = binge_velocity(df)
df[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "binge_velocity"]].head(3)

,netflix_title,netflix_viewing_hours,netflix_weeks,binge_velocity
0,Show 000,48840931.0,13,3.756995e+06
1,Show 001,68780466.0,9,7.642274e+06
2,Show 002,60617686.0,9,6.735298e+06


## 5. 值得信賴的評分：把小樣本向平均值收縮

只有 12 票、評分 9.4 的作品，和有 200,000 票、評分 7.8 的作品，
所含的資訊量並不對等，但單純的平均數卻會把兩者一視同仁。這正是 IMDb
自家歷史上的「加權評分」公式所要解決的問題：把每個片名的評分與一個
先驗值（prior，例如整個資料集的平均評分）混合，混合權重取決於這個
評分背後有多少票數支撐。票數少的片名會被拉向平均值；票數多的片名
則能保留自己原本的分數。這樣得到的「品質」特徵，會比原始評分欄位
更站得住腳。


In [5]:
def bayesian_rating(frame: pd.DataFrame,
                     rating_col: str,
                     votes_col: str,
                     prior_votes: int = 500) -> pd.Series:
    """根據票數，把每個評分向資料集平均值收縮。

    weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C

    其中 R 是該片名自己的評分，v 是其票數，C 是整個資料集的
    平均評分，m 則是 `prior_votes` 這個「信心」門檻：v 遠小於 m
    的片名會被強力拉向 C，v 遠大於 m 的片名幾乎不會被拉動。
    """
    v = frame[votes_col]
    R = frame[rating_col]
    C = frame[rating_col].mean()
    m = prior_votes
    return ((v / (v + m)) * R + (m / (v + m)) * C).rename(f"{rating_col}_shrunk")


df["imdb_rating_shrunk"] = bayesian_rating(df, "imdb_averageRating", "imdb_numVotes")
df[["netflix_title", "imdb_averageRating", "imdb_numVotes", "imdb_rating_shrunk"]].dropna().head(3)

,netflix_title,imdb_averageRating,imdb_numVotes,imdb_rating_shrunk
0,Show 000,6.6,131664.0,6.600282
1,Show 001,4.8,82446.0,4.811299
2,Show 002,8.0,98769.0,7.993323


## 6. 影評與觀眾的一致性差距

TMDB 的 `vote_average` 與 IMDb 的 `averageRating` 都是 0 到 10 分的
量表，來源是不同但有所重疊的觀眾群體。兩者之間差距小，代表不同觀眾
族群普遍看法一致；差距大，則代表這部作品（或劇本）在不同族群眼中呈現出
截然不同的樣貌——可能是有爭議的、小眾的，或是評價兩極的。兩種情況都可能
爆紅，但賭注性質不同，而這個差距本身正是一個兩個資料來源單獨都給不出的
全新訊號。


In [6]:
def audience_alignment_gap(frame: pd.DataFrame,
                            col_a: str = "tmdb_vote_average",
                            col_b: str = "imdb_averageRating") -> pd.Series:
    """同一量表下，兩個評分來源之間的絕對差值。"""
    return (frame[col_a] - frame[col_b]).abs().rename("audience_alignment_gap")


df["audience_alignment_gap"] = audience_alignment_gap(df)
df[["netflix_title", "tmdb_vote_average", "imdb_averageRating", "audience_alignment_gap"]].dropna().head(3)

,netflix_title,tmdb_vote_average,imdb_averageRating,audience_alignment_gap
0,Show 000,7.4,6.6,0.8
1,Show 001,7.8,4.8,3.0
2,Show 002,5.0,8.0,3.0


## 7. 話題量：用對數轉換馴服偏斜的投票數

投票數（`tmdb_vote_count`、`imdb_numVotes`）橫跨好幾個數量級——
從寥寥幾票到數十萬票都有。如果原封不動地餵給線性模型，少數極端值會
壓過其他一切。`log1p`（即 log(1 + x)，在 0 時也安全）能把這個範圍
壓縮成模型真正能用的平滑「受關注程度」訊號，而這與*被接受的程度*
是兩回事。


In [7]:
def log_buzz(frame: pd.DataFrame, votes_col: str) -> pd.Series:
    """經過 Log1p 轉換的票數——一個壓縮過的「話題量」特徵。"""
    return np.log1p(frame[votes_col]).rename(f"{votes_col}_log")


df["imdb_buzz_log"] = log_buzz(df, "imdb_numVotes")
df[["netflix_title", "imdb_numVotes", "imdb_buzz_log"]].dropna().head(30)

,netflix_title,imdb_numVotes,imdb_buzz_log
0,Show 000,131664.0,11.788016
1,Show 001,82446.0,11.319911
2,Show 002,98769.0,11.500549
3,Show 003,283089.0,12.553520
4,Show 004,75331.0,11.229660
5,Show 005,193547.0,12.173281
7,Show 007,268478.0,12.500528
9,Show 009,86598.0,11.369044
10,Show 010,21836.0,9.991361
11,Show 011,294714.0,12.593764


## 8. 把所有東西組裝成一個特徵矩陣

把上面的範例包裝成單一個管線函式（pipeline function）。這正是你真正
會交給模型的形式：執行一次，得到一個乾淨的數值資料框，加上目標欄位，
就能直接交給模組 0 的 `SimpleLinearModel.fit()`，或任何 scikit-learn
的估計器（estimator）。


In [8]:
def build_feature_matrix(raw: pd.DataFrame) -> pd.DataFrame:
    """把原始的綜合資料集轉換成模型可用的特徵矩陣。

    串接上面所有的特徵函式。沒有 IMDb 比對結果的列，其 IMDb 衍生
    特徵會保留 NaN——留給下游決定要捨棄、填補，還是明確地為這種
    缺失情況建模。
    """
    frame = label_hits(raw, "netflix_viewing_hours", quantile=0.75)
    frame["binge_velocity"] = binge_velocity(frame)
    frame["imdb_rating_shrunk"] = bayesian_rating(frame, "imdb_averageRating", "imdb_numVotes")
    frame["audience_alignment_gap"] = audience_alignment_gap(frame)
    frame["imdb_buzz_log"] = log_buzz(frame, "imdb_numVotes")

    feature_columns = [
        "binge_velocity",
        "imdb_rating_shrunk",
        "audience_alignment_gap",
        "imdb_buzz_log",
        "tmdb_popularity",
    ]
    return frame[["netflix_title", "is_hit", *feature_columns]]


features = build_feature_matrix(df)
features.describe()

,binge_velocity,imdb_rating_shrunk,audience_alignment_gap,imdb_buzz_log,tmdb_popularity
count,3.000000e+02,266.000000,216.000000,266.000000,243.000000
mean,5.542163e+06,6.673530,1.741204,11.649581,204.333333
std,2.045585e+06,1.480197,1.170695,1.027040,112.786259
min,2.108027e+06,4.014917,0.000000,6.459904,5.300000
25%,3.546625e+06,5.403707,0.700000,11.279534,115.150000
50%,5.761116e+06,6.699815,1.600000,11.976276,202.000000
75%,7.323639e+06,7.897830,2.525000,12.361748,295.250000
max,8.977398e+06,9.279912,4.700000,12.608537,399.800000


## 9. 健全性檢查：每個因子是否真的與目標相關？

在信任一個特徵之前，先看看它與 `is_hit` 的相關性（correlation）。
這種方式無法捕捉到所有有用的非線性關係，但它是一個快速、通用的
初步篩選——一個相關性接近零、又沒有明確領域理由支持的特徵，
就很值得考慮捨棄或重新思考。


In [9]:
numeric = features.drop(columns=["netflix_title"]).astype(float)
numeric.corr()["is_hit"].drop("is_hit").sort_values(key=abs, ascending=False)


binge_velocity            0.513524
tmdb_popularity           0.100953
audience_alignment_gap    0.022144
imdb_buzz_log             0.013298
imdb_rating_shrunk        0.005193
Name: is_hit, dtype: float64

## 接下來的方向

這些因子只是輸入，並不是模型本身——真正學習它們權重的，是模組 0 的
`SimpleLinearModel`（如果你要預測的是 `is_hit` 而非原始時數，也可以用
scikit-learn 的 `LogisticRegression`）。這份筆記本教你的，是一種紀律：
把一份雜亂、只有部分成功比對的綜合資料集，轉化成一小組有充分理由、
可重複使用的數值特徵，每一個都能追溯回一個關於「Netflix 電視劇為何成功」
的具體假設——而這正是你回頭審視自己劇本前提時，最值得帶著走的那種思考方式。
